# Predicting Housing Prices in Buenos Aires

This notebook develops and evaluates price models, with Capital Federal as the primary market.

## 1. Import Libraries

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid")

## 2. Import and Combine Data

Load both listing extracts and preserve their shared schema.

In [ ]:
data_dir = Path("data")
paths = sorted(data_dir.glob("buenos-aires-real-estate-*.csv"))

raw = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
print(f"Files: {len(paths)}")
print(f"Rows: {len(raw):,}")
print(f"Columns: {raw.shape[1]}")

In [ ]:
raw.head()

## 3. Validate the Raw Data

Check schema consistency, duplicates, operation type, geography, and target availability.

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate listing URLs:", raw["properati_url"].duplicated().sum())
print("Operations:\n", raw["operation"].value_counts(dropna=False))
print("\nMissing target values:", raw["price_aprox_usd"].isna().sum())
print("\nTop regions:\n", raw["place_with_parent_names"].dropna().str.strip("|").str.split("|").str[2].value_counts())

## 4. Clean and Engineer Features

Keep valid sale listings with a positive USD price. Optional listing fields remain available for pipeline imputation.

In [ ]:
df = raw.drop_duplicates().copy()

numeric_columns = [
    "price", "price_aprox_local_currency", "price_aprox_usd",
    "surface_total_in_m2", "surface_covered_in_m2", "price_usd_per_m2",
    "price_per_m2", "floor", "rooms", "expenses"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.loc[
    df["operation"].eq("sell")
    & df["price_aprox_usd"].notna()
    & (df["price_aprox_usd"] > 0)
].copy()

for column in ["surface_total_in_m2", "surface_covered_in_m2", "rooms", "expenses"]:
    df.loc[df[column] <= 0, column] = np.nan
df.loc[df["floor"] <= 0, "floor"] = np.nan

invalid_area = (
    df["surface_total_in_m2"].notna()
    & df["surface_covered_in_m2"].notna()
    & (df["surface_covered_in_m2"] > df["surface_total_in_m2"])
)
df.loc[invalid_area, "surface_covered_in_m2"] = np.nan

parts = df["place_with_parent_names"].fillna("").str.strip("|").str.split("|")
df["province"] = parts.str[0]
df["region"] = parts.str[1]
df["municipality"] = parts.str[2]
df["neighborhood"] = parts.str[3].fillna(parts.str[2])

coordinates = df["lat-lon"].str.split(",", n=1, expand=True)
df["latitude"] = pd.to_numeric(coordinates[0], errors="coerce")
df["longitude"] = pd.to_numeric(coordinates[1], errors="coerce")

df["log_price_usd"] = np.log1p(df["price_aprox_usd"])
print(f"Cleaned rows: {len(df):,}")

### Missingness profile

Missing values are quantified here and handled later inside model pipelines.

In [ ]:
missingness = (
    df.isna().mean().mul(100).sort_values(ascending=False).rename("missing_percent").to_frame()
)
missingness.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
plot_missing = missingness.head(12).sort_values("missing_percent")
sns.barplot(x="missing_percent", y=plot_missing.index, data=plot_missing, color="darkorange")
plt.title("Missing Values in the Cleaned Dataset")
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.tight_layout()

## 5. Define the Primary Market

Capital Federal is the primary modeling cohort. The wider market remains available for context and out-of-scope robustness checks.

In [ ]:
capital = df.loc[df["region"].eq("Capital Federal")].copy()
non_capital = df.loc[~df["region"].eq("Capital Federal")].copy()

print(f"Capital Federal listings: {len(capital):,}")
print(f"Other Buenos Aires listings: {len(non_capital):,}")
print("\nCapital Federal property types:\n", capital["property_type"].value_counts())
print("\nCapital Federal neighborhoods:", capital["neighborhood"].nunique())

## 6. Exploratory Data Analysis

Use EDA to understand skew, market composition, size relationships, and neighborhood variation before modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(capital["price_aprox_usd"], bins=50, ax=axes[0])
axes[0].set_title("Capital Federal Listing Prices")
axes[0].set_xlabel("Price (USD)")
sns.histplot(capital["log_price_usd"], bins=50, ax=axes[1])
axes[1].set_title("Log Price Distribution")
axes[1].set_xlabel("log(Price + 1)")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = capital["property_type"].value_counts().index
sns.countplot(data=capital, y="property_type", order=order, ax=axes[0])
axes[0].set_title("Capital Federal Listings by Property Type")

price_plot = capital.assign(
    capped_price=capital["price_aprox_usd"].clip(upper=capital["price_aprox_usd"].quantile(0.99))
)
sns.boxplot(data=price_plot, x="property_type", y="capped_price", ax=axes[1])
axes[1].set_title("Price by Property Type")
axes[1].set_xlabel("")
axes[1].set_ylabel("Price (USD; capped at 99th percentile)")
plt.tight_layout()

In [ ]:
plot_size = capital[["surface_total_in_m2", "price_aprox_usd", "property_type"]].dropna()
plot_size = plot_size.loc[plot_size["surface_total_in_m2"] > 0]
plot_size = plot_size.loc[plot_size["price_aprox_usd"] <= plot_size["price_aprox_usd"].quantile(0.99)]
plot_size = plot_size.sample(min(2500, len(plot_size)), random_state=42)

sns.scatterplot(data=plot_size, x="surface_total_in_m2", y="price_aprox_usd", hue="property_type", alpha=0.6)
plt.title("Price versus Total Surface Area")
plt.xlabel("Total surface area (m²)")
plt.ylabel("Price (USD)")
plt.tight_layout()

In [ ]:
neighborhood_summary = (
    capital.groupby("neighborhood")
    .agg(listings=("price_aprox_usd", "size"), median_price=("price_aprox_usd", "median"))
    .query("listings >= 20")
    .sort_values("median_price", ascending=False)
    .head(15)
    .sort_values("median_price")
)

plt.figure(figsize=(10, 6))
sns.barplot(data=neighborhood_summary, x="median_price", y=neighborhood_summary.index, color="steelblue")
plt.title("Highest-Median-Price Capital Federal Neighborhoods")
plt.xlabel("Median price (USD)")
plt.ylabel("Neighborhood")
plt.tight_layout()

## 7. Define Modeling Features

Target-derived price-per-square-metre fields and listing URLs are excluded to prevent leakage and identifier memorization.

In [ ]:
target = "price_aprox_usd"
leakage_columns = ["price_usd_per_m2", "price_per_m2", "log_price_usd"]
feature_columns = [
    "property_type", "neighborhood", "municipality",
    "surface_total_in_m2", "surface_covered_in_m2", "floor", "rooms",
    "expenses", "latitude", "longitude"
]

X = capital[feature_columns].copy()
y = capital[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print("Excluded from predictors:", leakage_columns + ["properati_url"])

## 8. Establish Baselines

Compare a global median with a neighborhood/property-type median. These baselines set a practical minimum for model performance.

In [ ]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": np.mean(np.abs((actual - predicted) / actual)) * 100,
        "R2": r2_score(actual, predicted),
    }

global_median = y_train.median()
global_pred = np.full(len(y_test), global_median)

train_groups = pd.DataFrame({"neighborhood": X_train["neighborhood"], "property_type": X_train["property_type"], "target": y_train})
group_medians = train_groups.groupby(["neighborhood", "property_type"])["target"].median()

def group_baseline(frame):
    values = []
    for _, row in frame.iterrows():
        key = (row["neighborhood"], row["property_type"])
        values.append(group_medians.get(key, global_median))
    return np.asarray(values)

group_pred = group_baseline(X_test)
baseline_results = pd.DataFrame([
    {"model": "Global median", "target": "USD", **regression_metrics(y_test, global_pred)},
    {"model": "Neighborhood + type median", "target": "USD", **regression_metrics(y_test, group_pred)},
])
baseline_results

## 9. Build Preprocessing Pipelines

Imputation, scaling, and encoding are fitted only on training data through a shared preprocessing pipeline.

In [ ]:
numeric_features = [
    "surface_total_in_m2", "surface_covered_in_m2", "floor", "rooms",
    "expenses", "latitude", "longitude"
]
categorical_features = ["property_type", "neighborhood", "municipality"]

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False)),
        ]),
        categorical_features,
    ),
], remainder="drop")

## 10. Compare Models and Target Scales

Evaluate ordinary and regularized linear models alongside tree ensembles. Raw-price and log-price targets are compared on the same USD test set.

In [ ]:
models = {
    "Linear regression": LinearRegression(),
    "Ridge": Ridge(alpha=10.0),
    "Lasso": Lasso(alpha=100.0, max_iter=20_000),
    "Random forest": RandomForestRegressor(
        n_estimators=150, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "Gradient boosting": GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42
    ),
}

def fit_and_score(model, target_name):
    target_train = np.log1p(y_train) if target_name == "log-USD" else y_train
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, target_train)
    prediction = pipeline.predict(X_test)
    if target_name == "log-USD":
        prediction = np.expm1(prediction)
    return pipeline, regression_metrics(y_test, prediction), prediction

results = []
fits = {}
for target_name in ["USD", "log-USD"]:
    for model_name, estimator in models.items():
        fitted, metrics, prediction = fit_and_score(estimator, target_name)
        fits[(model_name, target_name)] = (fitted, prediction)
        results.append({"model": model_name, "target": target_name, **metrics})

model_results = pd.DataFrame(results).sort_values("MAE")
model_results

In [ ]:
comparison = pd.concat([baseline_results, model_results], ignore_index=True)
comparison.sort_values("MAE")

## 11. Select and Inspect the Best Model

Select the lowest-MAE model, then inspect errors and performance across property types and price bands.

In [ ]:
best_row = model_results.sort_values("MAE").iloc[0]
best_key = (best_row["model"], best_row["target"])
best_pipeline, best_prediction = fits[best_key]

print(f"Selected model: {best_key[0]} ({best_key[1]} target)")
print(f"Test MAE: ${best_row['MAE']:,.0f}")
print(f"Test RMSE: ${best_row['RMSE']:,.0f}")
print(f"Test MAPE: {best_row['MAPE']:.1f}%")
print(f"Test R²: {best_row['R2']:.3f}")

In [ ]:
residuals = pd.DataFrame({
    "actual": y_test,
    "predicted": best_prediction,
    "error": y_test.to_numpy() - best_prediction,
    "property_type": X_test["property_type"].to_numpy(),
    "neighborhood": X_test["neighborhood"].to_numpy(),
})
residuals["absolute_error"] = residuals["error"].abs()
residuals["price_band"] = pd.qcut(residuals["actual"], q=4, duplicates="drop")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=residuals, x="actual", y="predicted", alpha=0.5, ax=axes[0])
max_value = max(residuals["actual"].max(), residuals["predicted"].max())
axes[0].plot([0, max_value], [0, max_value], "r--")
axes[0].set_title("Actual versus Predicted Prices")
axes[0].set_xlabel("Actual price (USD)")
axes[0].set_ylabel("Predicted price (USD)")

sns.histplot(residuals["error"], bins=40, ax=axes[1])
axes[1].set_title("Prediction Errors")
axes[1].set_xlabel("Actual - predicted (USD)")
plt.tight_layout()

In [ ]:
error_by_type = residuals.groupby("property_type").agg(
    listings=("actual", "size"),
    MAE=("absolute_error", "mean"),
    median_error=("error", "median"),
).sort_values("MAE", ascending=False)

error_by_band = residuals.groupby("price_band", observed=True).agg(
    listings=("actual", "size"),
    MAE=("absolute_error", "mean"),
    median_error=("error", "median"),
)

print("Error by property type")
display(error_by_type)
print("Error by actual-price quartile")
display(error_by_band)

## 12. Robustness Outside Capital Federal

Evaluate the selected Capital Federal model on other cleaned Buenos Aires listings to show where geographic generalization weakens.

In [ ]:
X_external = non_capital[feature_columns].copy()
y_external = non_capital[target].copy()
external_prediction = best_pipeline.predict(X_external)
if best_key[1] == "log-USD":
    external_prediction = np.expm1(external_prediction)
external_metrics = regression_metrics(y_external, external_prediction)
print(f"External listings: {len(X_external):,}")
print(external_metrics)

## 13. Findings and Next Steps

The executed outputs above provide the measured model comparison and error analysis for the README. The next stage is to document the selected model, its limitations, and any deployment or monitoring requirements. No further modeling is performed in this notebook.